# Model Experimentation Notebook

In this notebook, we experiment with multiple baseline models (without hyperparameter tuning)
to identify which algorithm performs best on our fraud detection problem.

**Important:** Since our data is highly imbalanced (0.13% fraud), we will:
- Use `class_weight='balanced'` in our models instead of SMOTE
- Evaluate using Precision, Recall, F1-score, and ROC-AUC — NOT accuracy (which would be misleading)

## Step 1: Load Train/Test Data

We load the pre-split, feature-engineered train and test sets created earlier by our pipeline scripts (`build_features.py` and `split_data.py`).

In [2]:
import pandas as pd

# Load the train/test splits saved earlier
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Fraud ratio in y_train:", y_train.mean())
print("Fraud ratio in y_test:", y_test.mean())

X_train shape: (5090096, 11)
X_test shape: (1272524, 11)
Fraud ratio in y_train: 0.0012907418642005967
Fraud ratio in y_test: 0.0012911347840983745


## Step 2 Baseline Model — Logistic Regression (with MLflow Tracking)

We re-run our Logistic Regression baseline, this time logging parameters,
metrics, and the trained model to MLflow — so we can compare all experiments
in one place later using `mlflow ui`.

In [9]:
import mlflow

# Use SQLite backend instead of file-based storage (avoids maintenance mode error)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [10]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, precision_score, recall_score, f1_score

# Set experiment name (creates it if it doesn't exist)
mlflow.set_experiment("fraud-detection-baseline-models")

with mlflow.start_run(run_name="logistic_regression_baseline"):
    # Train model
    log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    log_reg.fit(X_train, y_train)

    # Predictions
    y_pred = log_reg.predict(X_test)
    y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)

    # Log metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # Log the trained model itself
    mlflow.sklearn.log_model(log_reg, "model")

    # Print results as before
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("ROC-AUC Score:", roc_auc)

2026/07/30 17:35:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/30 17:35:33 INFO mlflow.store.db.utils: Updating database tables
2026/07/30 17:35:38 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-baseline-models' does not exist. Creating a new experiment.
2026/07/30 17:36:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97   1270881
           1       0.02      0.98      0.04      1643

    accuracy                           0.94   1272524
   macro avg       0.51      0.96      0.50   1272524
weighted avg       1.00      0.94      0.97   1272524

Confusion Matrix:
[[1190032   80849]
 [     35    1608]]
ROC-AUC Score: 0.99068662445439


## Step 3: Baseline Model — Random Forest (with MLflow Tracking)

We now train Random Forest as our second baseline model, logging it to the
same MLflow experiment so we can compare it against Logistic Regression.

In [11]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="random_forest_baseline"):

    # Train model
    rf_model = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        max_depth=10
    )
    rf_model.fit(X_train, y_train)

    # Predictions
    y_pred_rf = rf_model.predict(X_test)
    y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision_rf = precision_score(y_test, y_pred_rf)
    recall_rf = recall_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)
    roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)

    # Log metrics
    mlflow.log_metric("precision", precision_rf)
    mlflow.log_metric("recall", recall_rf)
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)

    # Log the trained model
    mlflow.sklearn.log_model(rf_model, "model")

    # Print results
    print("Classification Report (Random Forest):")
    print(classification_report(y_test, y_pred_rf))
    print("Confusion Matrix (Random Forest):")
    print(confusion_matrix(y_test, y_pred_rf))
    print("ROC-AUC Score (Random Forest):", roc_auc_rf)

2026/07/30 17:47:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       1.00      0.99      0.99   1270881
           1       0.08      0.99      0.15      1643

    accuracy                           0.99   1272524
   macro avg       0.54      0.99      0.57   1272524
weighted avg       1.00      0.99      0.99   1272524

Confusion Matrix (Random Forest):
[[1251868   19013]
 [     10    1633]]
ROC-AUC Score (Random Forest): 0.9992764279660398


## Step 4: Baseline Model — XGBoost (with MLflow Tracking)

We now try XGBoost, a gradient boosting algorithm that is widely considered
one of the best performers for imbalanced classification problems like fraud detection.

Instead of `class_weight='balanced'`, XGBoost uses `scale_pos_weight` — a ratio
that tells the model how much more to focus on the minority (fraud) class.

In [13]:
import mlflow.xgboost

In [14]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight: ratio of negative class to positive class
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight value:", scale_pos_weight)

with mlflow.start_run(run_name="xgboost_baseline"):

    # Train model
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)

    # Predictions
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision_xgb = precision_score(y_test, y_pred_xgb)
    recall_xgb = recall_score(y_test, y_pred_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

    # Log parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 6)

    # Log metrics
    mlflow.log_metric("precision", precision_xgb)
    mlflow.log_metric("recall", recall_xgb)
    mlflow.log_metric("f1_score", f1_xgb)
    mlflow.log_metric("roc_auc", roc_auc_xgb)

    # Log the trained model
    mlflow.xgboost.log_model(xgb_model, "model")

    # Print results
    print("Classification Report (XGBoost):")
    print(classification_report(y_test, y_pred_xgb))
    print("Confusion Matrix (XGBoost):")
    print(confusion_matrix(y_test, y_pred_xgb))
    print("ROC-AUC Score (XGBoost):", roc_auc_xgb)

scale_pos_weight value: 773.7482496194825


2026/07/30 17:58:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Classification Report (XGBoost):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.52      0.99      0.68      1643

    accuracy                           1.00   1272524
   macro avg       0.76      0.99      0.84   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion Matrix (XGBoost):
[[1269351    1530]
 [     17    1626]]
ROC-AUC Score (XGBoost): 0.9994933173494438
